# FIT5196 Assessment 1 — Group030 Solution
Structured, reproducible transformation and validation workflow.

## 0. Configuration and dependencies

In [ ]:
from pathlib import Path
GROUP_ID="Group030"
INPUT_DIR=Path("raw_input")
OUTPUT_DIR=Path("outputs")
DICTIONARY_PATH=Path("public_data_dictionary.csv")
OUTPUT_DIR.mkdir(exist_ok=True)
import pandas as pd
from Group030_solution import build_tables, validate
from Group030_text_functions import *

## 1. Structured source profiling
JSON is loaded with `json.load`; XML is loaded with `ElementTree`. Regex is only applied after narrative values are selected. JSON collections: customerProfiles (customer grain), orders with repeated shoppingCart (order/item grains) and productReviews (review grain). XML repeats Order/Item, Product and Review elements. Candidate keys match the public dictionary. JSON uses ISO dates, booleans and numeric currency; XML uses day-first dates, Y/N and AUD-labelled currency. Empty strings/elements represent missing source strings.

In [ ]:
tables, profile = build_tables(INPUT_DIR, DICTIONARY_PATH)
profile

## 2. Source-to-target mapping
Every target field is documented in `Group030_source_to_target_mapping.csv` with stable MAP IDs.

In [ ]:
mapping=pd.read_csv("Group030_source_to_target_mapping.csv",keep_default_na=False)
mapping.groupby("output_table").agg(rows=("mapping_id","size"),complete=("source_format",lambda x:(x!="").sum()))

## 3. Text and regex functions

In [ ]:
import csv
cases=pd.read_csv("templates/A1_public_text_test_cases.csv",keep_default_na=False)
results=[]
for _,r in cases.iterrows():
    actual=str(globals()[r.function](r.input_value))
    results.append((r.case_id,actual,r.expected_output,actual==r.expected_output))
pd.DataFrame(results,columns=["case_id","actual","expected","pass"])

Student-designed edge cases include missing inputs, Unicode Latin diacritics, all-non-Latin reviews and malformed embedded/extended references.

In [ ]:
assert clean_narrative_text(None)=="NaN"
assert build_latin_analysis("déjà 東京")=="déjà"
assert contains_non_latin_script("déjà 東京") is True
assert extract_order_reference("XHORD123456")=="NaN"
assert extract_product_sku("SKU-ABC_extra")=="NaN"

## 4. Build six relational tables
`build_tables` flattens arrays at entity grain, normalises both sources and calculates published arithmetic. The submitted `Group030_solution.py` is the notebook export/maintained implementation.

In [ ]:
{name: {"rows":len(df),"columns":len(df.columns)} for name,df in tables.items()}

## 5. Reconciliation
Normalised records are compared by stable primary key. Equal overlap is collapsed; differing non-missing values are recorded as conflicts rather than resolved by arbitrary source precedence.

In [ ]:
{"normalised_conflicts":len(profile["conflicts"]),"details":profile["conflicts"][:5]}

## 6. Validation register

In [ ]:
dictionary=pd.read_csv(DICTIONARY_PATH)
validation_register=validate(tables,dictionary,profile)
validation_register

The register gives stable IDs, observed results, PASS/FAIL and interpretations for schema/order, keys, relationships, overlap, arithmetic, temporal logic, missing sentinels and multilingual preservation.

## 7. Export and final reproducibility record

In [ ]:
for name,df in tables.items():
    df.to_csv(OUTPUT_DIR/f"{GROUP_ID}_{name}_standardised.csv",index=False,na_rep="NaN")
validation_register.to_csv(OUTPUT_DIR/f"{GROUP_ID}_validation_register.csv",index=False)
assert set(validation_register.status)=={"PASS"}
print("Fresh offline run complete; all six outputs recreated.")